# Short-Term Memory in LLM Agents

This notebook demonstrates how short-term memory works in conversational agents using LangChain and OpenAI (`gpt-6-luna`).

*By Shivank Agarwal*

**Topics covered:**
1. Basic conversation with memory buffer
2. Trimming to manage context length
3. Deletion to manage context length
4. Summarization to manage context length

## Setup

In [ ]:
!pip install -q langchain langchain-openai langgraph

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# reasoning_effort="none": answer straight away, no hidden "thinking" tokens
llm = ChatOpenAI(model="gpt-6-luna", reasoning_effort="none")

---
## 1. Basic Conversation Buffer Memory

The simplest form of memory - stores all messages in a buffer.

In [ ]:
# Simple message buffer (list of messages)
message_buffer = []

def chat(user_input):
    """Chat function that maintains conversation history."""
    message_buffer.append(HumanMessage(content=user_input))
    response = llm.invoke(message_buffer)
    message_buffer.append(response)
    return response.content

def dump_memory():
    """Display all messages in the buffer."""
    print("=" * 50)
    print("MEMORY DUMP")
    print("=" * 50)
    for i, msg in enumerate(message_buffer):
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"[{i}] {role}: {msg.content[:100]}{'...' if len(msg.content) > 100 else ''}")
    print(f"\nTotal messages: {len(message_buffer)}")
    print("=" * 50)

In [ ]:
# 5-turn conversation
conversations = [
    "Hi! My name is Alice and I'm learning about AI.",
    "What are the main types of machine learning?",
    "Can you explain supervised learning with an example?",
    "What's the difference between classification and regression?",
    "Thanks! Can you remind me what my name is?"
]

for user_msg in conversations:
    print(f"Human: {user_msg}")
    response = chat(user_msg)
    print(f"AI: {response}\n")

In [ ]:
# Full memory dump - all 10 messages (5 human + 5 AI)
dump_memory()

**Key Observation:** The buffer stores ALL messages. The AI can recall the user's name from turn 1 because the full history is preserved.

**Problem:** As conversations grow, the buffer can exceed the model's context window.

---
## 2. Trimming - Keep Last K Messages

Trimming removes the **oldest** messages when the buffer exceeds a limit. It's a sliding window approach.

In [ ]:
from langchain_core.messages import trim_messages

# Reset buffer
message_buffer = []

# Rebuild conversation
for user_msg in conversations:
    message_buffer.append(HumanMessage(content=user_msg))
    response = llm.invoke(message_buffer)
    message_buffer.append(response)

print("BEFORE TRIMMING:")
dump_memory()

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

# Keep only the most recent messages that fit in ~300 tokens
trimmed_messages = trim_messages(
    message_buffer,
    max_tokens=300,
    strategy="last",                          # keep the LAST messages, drop the oldest
    token_counter=count_tokens_approximately, # fast token estimate (about 4 characters per token)
    start_on="human",                         # the kept history must start with a user message
    allow_partial=False,                      # never cut a message in half
)

print("AFTER TRIMMING (last ~300 tokens worth of messages):")
print("=" * 50)
for i, msg in enumerate(trimmed_messages):
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"[{i}] {role}: {msg.content[:80]}...")
print(f"\nMessages retained: {len(trimmed_messages)} (was {len(message_buffer)})")
print("=" * 50)

**Key Observation:** Trimming removes old messages entirely. Early context (like the user's name) is LOST.

---
## 3. Deletion - Selective Message Removal

Unlike trimming (which removes oldest messages), deletion allows **targeted removal** of specific messages.

In [ ]:
# Reset and rebuild
message_buffer = []
for user_msg in conversations:
    message_buffer.append(HumanMessage(content=user_msg))
    response = llm.invoke(message_buffer)
    message_buffer.append(response)

print("ORIGINAL BUFFER:")
dump_memory()

In [ ]:
# DELETION: Remove specific messages (e.g., the long explanations in middle)
# We'll delete messages 4-7 (the detailed ML explanations) but keep first and last

indices_to_delete = [4, 5, 6, 7]  # Middle messages with long explanations

deleted_buffer = [msg for i, msg in enumerate(message_buffer) if i not in indices_to_delete]

print("AFTER DELETION (removed indices 4-7):")
print("=" * 50)
for i, msg in enumerate(deleted_buffer):
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"[{i}] {role}: {msg.content[:80]}{'...' if len(msg.content) > 80 else ''}")
print(f"\nMessages retained: {len(deleted_buffer)} (was {len(message_buffer)})")
print("=" * 50)

In [ ]:
# Compare: Can we still recall the user's name?
# With TRIMMING - name was lost (oldest messages removed)
# With DELETION - name preserved (we kept first messages)

deleted_buffer.append(HumanMessage(content="What's my name again?"))
response = llm.invoke(deleted_buffer)
print(f"AI Response: {response.content}")
print("\n** The name 'Alice' is preserved because we deleted MIDDLE messages, not the beginning! **")

### Deletion inside a LangGraph agent

In a LangGraph agent the history lives in the graph's **state**, and you delete by returning `RemoveMessage(id=...)`. LangGraph's `add_messages` reducer (the same one `MessagesState` uses) then removes those messages permanently.

In [ ]:
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import add_messages

state_messages = add_messages([], message_buffer)          # gives every message an id, as LangGraph state does
print(f"Before: {len(state_messages)} messages")

remove_first_two = [RemoveMessage(id=m.id) for m in state_messages[:2]]
state_messages = add_messages(state_messages, remove_first_two)
print(f"After removing the earliest two: {len(state_messages)} messages")
print("First message now:", state_messages[0].content[:80])

### Trimming vs Deletion - Key Differences

| Aspect | Trimming | Deletion |
|--------|----------|----------|
| What's removed | Oldest messages (FIFO) | Specific messages by index/criteria |
| Early context | LOST | Can be PRESERVED |
| Use case | Simple sliding window | Selective memory management |
| Control | Automatic | Manual/rule-based |

---
## 4. Summarization - Compress Old Context

Instead of losing information, summarization **condenses** old messages into a summary.

In [ ]:
def summarize_messages(messages, llm):
    """Summarize a list of messages into a single summary."""
    conversation_text = "\n".join([
        f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
        for m in messages
    ])
    
    summary_prompt = f"""Summarize this conversation in 2-3 sentences, preserving key facts:

{conversation_text}

Summary:"""
    
    summary = llm.invoke([HumanMessage(content=summary_prompt)])
    return summary.content

In [ ]:
# Reset and rebuild
message_buffer = []
for user_msg in conversations:
    message_buffer.append(HumanMessage(content=user_msg))
    response = llm.invoke(message_buffer)
    message_buffer.append(response)

print("ORIGINAL BUFFER:")
dump_memory()

In [ ]:
# Summarize older messages (first 6), keep recent (last 4)
old_messages = message_buffer[:6]
recent_messages = message_buffer[6:]

summary = summarize_messages(old_messages, llm)
print("SUMMARY OF OLD MESSAGES:")
print(summary)

In [ ]:
# Create new buffer with summary + recent messages
summarized_buffer = [
    SystemMessage(content=f"Previous conversation summary: {summary}")
] + recent_messages

print("SUMMARIZED BUFFER:")
print("=" * 50)
for i, msg in enumerate(summarized_buffer):
    if isinstance(msg, SystemMessage):
        role = "System"
    elif isinstance(msg, HumanMessage):
        role = "Human"
    else:
        role = "AI"
    print(f"[{i}] {role}: {msg.content[:100]}{'...' if len(msg.content) > 100 else ''}")
print(f"\nMessages: {len(summarized_buffer)} (was {len(message_buffer)})")
print("=" * 50)

In [ ]:
# Test: Can we still recall the user's name?
summarized_buffer.append(HumanMessage(content="What's my name?"))
response = llm.invoke(summarized_buffer)
print(f"AI Response: {response.content}")
print("\n** The name is preserved in the SUMMARY even though original messages were compressed! **")

---
## Summary: Memory Management Strategies

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| **Full Buffer** | Complete context | Can exceed token limits | Short conversations |
| **Trimming** | Simple, automatic | Loses early context | Recency-focused tasks |
| **Deletion** | Selective control | Requires logic to decide | Important info scattered |
| **Summarization** | Preserves key info | Adds LLM calls, may lose detail | Long conversations |